### Model Pipeline

1. Raw table_data (string)

        ↓ linearize

2. Linearized text (string)

        ↓ tokenizer.encode()

3. Token IDs (integers)

        ↓ model's embedding layer (learned lookup table, inside the model)

4. Token embeddings + positional embeddings

        ↓ encoder self-attention layers (bidirectional)

5. Encoder hidden states (contextualized representations, one vector per input token)

        ↓ fed into every decoder layer via cross-attention

6. Decoder (causal self-attention + cross-attention into step 5)

        ↓ generates one token at a time, autoregressively

7. Output logits → softmax → generated token

        ↓ repeat step 6-7 until <eos>
        
8. Decoded text = generated financial commentary

### Installs and Imports

In [2]:
!pip install -q -U transformers
!pip install -q -U datasets
!pip install -q -U evaluate
!pip install -q "tokenizers>=0.22.0,<0.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [3]:
!pip install  -q -U rouge_score
!pip install  -q -U nltk
!pip install  -q -U sacrebleu
!pip install  -q -U bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00


In [4]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
import evaluate
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
import nltk

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [5]:
def init_seed(init_random=20):
  random.seed(init_random)
  np.random.seed(init_random)
  torch.manual_seed(init_random)
def sys_info():
  print("PyTorch version:", torch.__version__)
  if torch.cuda.is_available():
    print("GPU Present :", torch.cuda.get_device_name(0))
  else:
    print("CUDA not available !")


### Setting Execution environment

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# path setup
path1 = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/train_splitEDA.py"
path2 = "/content/drive/MyDrive/GitHub/266-final-project/train_splitEDA.py"
prjfolder = "/content/drive/MyDrive/GitHub/266-final-project/"
if os.path.exists(path1):
  prjfolder = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/"
elif os.path.exists(path2):
  prjfolder = "/content/drive/MyDrive/GitHub/266-final-project/"
prjfolder
save_dir = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/"

In [9]:
initseed = 20
Max_input_length = 1024 # fixed increased from 256
Max_target_length = 1024 # fixed increased from 384
init_seed(initseed)
sys_info()
init_script = os.path.join(prjfolder, "train_splitEDA.py")
init_script

PyTorch version: 2.11.0+cpu
CUDA not available !


'/content/drive/MyDrive/GitHub/266-final-project/train_splitEDA.py'

In [10]:
%%capture
%run -i "$init_script"

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7823 > 512). Running this sequence through the model will result in indexing errors


### Data preparation / formatting

In [11]:
def build_encoder_input(row):
    return (
        "Generate a financial market report.\n\n"
        f"Instruction:\n"
        f"{row['instruction']}\n\n"
        f"Market summary:\n"
        f"{row['market_summary']}"
    )
def prep_data(df):
  df["market_summary"] = df["table_data"].apply(summarize_market_table) # fixing order of market_summary col creation
  df["encoder_input"] = df.apply(build_encoder_input, axis=1)
  df["decoder_target"] = df["report"]
  df["input_length"] = df["encoder_input"].apply(token_length)
  df["target_length"] = df["decoder_target"].apply(token_length)
  return df
def print_ip_op(df, index=0):
  print("Input:",df.iloc[index]["encoder_input"])
  print("Output:",df.iloc[index]["decoder_target"])
  return
def count_percent_by_threshold(df, col, threshold):
  count = (df[col] > threshold).sum()
  percent = (df[col] > threshold).mean()
  print(f'{col} > {threshold}: {count} ({percent:.2%})')
  return

In [12]:
summarize_market_table(train_df.iloc[0]["table_data"])

'Product: Live Cattle Future (front month) (December)\nSymbol: LEZ1\nPeriod start: 2021-11-02\nPeriod end: 2021-12-02\nTrading days: 132\nStarting close: 129.95\nEnding close: 170.90\nPeriod high: 171.00\nPeriod low: 128.88\nAbsolute price change: 40.95\nPercentage price change: 31.51%\nDaily return volatility: 1.25%\nMaximum daily gain: 10.11%\nMaximum daily loss: -4.69%\nAverage volume: 9696\nMaximum volume: 31903\nMinimum volume: 558\nOverall price trend: Upward'

In [13]:
train_df = prep_data(train_df) # fix to assign back return value to df



In [14]:
print_ip_op(train_df, 0)

Input: Generate a financial market report.

Instruction:
Please act as an expert financial market analyst. Please generate a market report:
1. by analyzing the historical market data provided.
2. following the market report example provided.

Market summary:
Product: Live Cattle Future (front month) (December)
Symbol: LEZ1
Period start: 2021-11-02
Period end: 2021-12-02
Trading days: 132
Starting close: 129.95
Ending close: 170.90
Period high: 171.00
Period low: 128.88
Absolute price change: 40.95
Percentage price change: 31.51%
Daily return volatility: 1.25%
Maximum daily gain: 10.11%
Maximum daily loss: -4.69%
Average volume: 9696
Maximum volume: 31903
Minimum volume: 558
Overall price trend: Upward
Output: Cattle futures posted moderate to strong gains as cash trade stayed supportive in the live cattle market. Dec live cattle gained 1.650 to 137.650, and Feb cattle were .975 higher to 139.575. Feeders saw mixed, to mostly higher market as Jan feeders were slightly lower, losing .050

In [15]:
test_df = prep_data(test_df)

In [16]:
print_ip_op(test_df)

Input: Generate a financial market report.

Instruction:
Please act as an expert financial market analyst. Please generate a market report:
1. by analyzing the historical market data provided.
2. following the market report example provided.

Market summary:
Product: Live Cattle Future (front month) (October)
Symbol: LEV2
Period start: 2022-09-26
Period end: 2022-10-25
Trading days: 132
Starting close: 143.48
Ending close: 180.25
Period high: 182.38
Period low: 142.72
Absolute price change: 36.77
Percentage price change: 25.63%
Daily return volatility: 1.37%
Maximum daily gain: 13.01%
Maximum daily loss: -2.85%
Average volume: 8873
Maximum volume: 35921
Minimum volume: 510
Overall price trend: Upward
Output: Oct live cattle gained 0.075 to 151.675, closing with a new contract high for the seventh straight day, and Dec slipped 0.825 to 153.300. Feeders were mostly lower with the exception of the front month with expiration on the 27th. Nov feeders traded 1.225 lower to 177.925. December

In [17]:
train_dataset = Dataset.from_pandas(train_df[["encoder_input", "decoder_target"]],
                                    preserve_index=False)

test_dataset = Dataset.from_pandas(test_df[["encoder_input", "decoder_target"]],
                                   preserve_index=False)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['encoder_input', 'decoder_target'],
    num_rows: 3143
})
Dataset({
    features: ['encoder_input', 'decoder_target'],
    num_rows: 795
})


In [18]:
train_df[["input_length", "target_length"]].describe()

,input_length,target_length
count,3143.000000,3143.000000
mean,174.361120,131.379574
std,6.731297,86.968900
min,148.000000,6.000000
25%,172.000000,67.000000
50%,176.000000,113.000000
75%,178.000000,169.500000
max,188.000000,560.000000


In [19]:
test_df[["input_length", "target_length"]].describe()

,input_length,target_length
count,795.000000,795.000000
mean,174.855346,148.171069
std,6.412594,74.627747
min,149.000000,11.000000
25%,173.000000,96.000000
50%,176.000000,139.000000
75%,178.000000,184.000000
max,190.000000,424.000000


In [20]:
count_percent_by_threshold(train_df, "target_length", Max_target_length)

target_length > 1024: 0 (0.00%)


In [21]:
count_percent_by_threshold(train_df, "input_length", Max_input_length)

input_length > 1024: 0 (0.00%)


##BART Model Training and Evaluation

In [22]:
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# 1. Initialize BART Model and Tokenizer
bart_model_name = "facebook/bart-base"
tokenizer_bart = BartTokenizer.from_pretrained(bart_model_name)
model_bart = BartForConditionalGeneration.from_pretrained(bart_model_name)



vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [23]:
# 2. Tokenization Function for BART


def tokenize_batch_bart(examples):
    model_inputs = tokenizer_bart(
        examples["encoder_input"],
        max_length=Max_input_length,
        truncation=True
    )

    labels = tokenizer_bart(
        text_target=examples["decoder_target"],
        max_length=Max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs





In [24]:
# 3. Process Datasets
tokenized_train_bart = train_dataset.map(tokenize_batch_bart, batched=True)
tokenized_test_bart = test_dataset.map(tokenize_batch_bart, batched=True)


Map:   0%|          | 0/3143 [00:00<?, ? examples/s]

Map:   0%|          | 0/795 [00:00<?, ? examples/s]

Setup till here for loaded model

In [51]:
# 4. Data Collator and Training Arguments
data_collator_bart = DataCollatorForSeq2Seq(tokenizer=tokenizer_bart, model=model_bart)

training_args_bart = Seq2SeqTrainingArguments(
    output_dir="./bart-financial-commentary",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)


In [52]:
# 5. Initialize Trainer
trainer_bart = Seq2SeqTrainer(
    model=model_bart,
    args=training_args_bart,
    train_dataset=tokenized_train_bart,
    eval_dataset=tokenized_test_bart,
    processing_class=tokenizer_bart,
    data_collator=data_collator_bart
)

print("BART Trainer initialized. Run trainer_bart.train() to begin training.")

BART Trainer initialized. Run trainer_bart.train() to begin training.


In [46]:
# 6 Training
trainer_bart.train()

Epoch,Training Loss,Validation Loss
1,No log,2.578528
2,3.006525,2.488729
3,2.543632,2.436686
4,2.382513,2.412053
5,2.382513,2.409544


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1965, training_loss=2.5614477725429388, metrics={'train_runtime': 550.8229, 'train_samples_per_second': 28.53, 'train_steps_per_second': 3.567, 'total_flos': 1979274806046720.0, 'train_loss': 2.5614477725429388, 'epoch': 5.0})

In [47]:
# evaluate on test set
test_metrics = trainer_bart.evaluate(
    max_length=Max_target_length,
    num_beams=4
)

print(test_metrics)

Training Loss,Validation Loss,Epoch
2.382513,2.409544,5


{'eval_loss': 2.4095442295074463}


### Save BART Model and Tokenizer
This cell saves the fine-tuned BART model and tokenizer to a persistent directory on Google Drive.

In [48]:
import os

# Define the save path for BART
bart_model_dir = os.path.join(save_dir, "bart_base_model1")

# Save model and tokenizer
trainer_bart.save_model(bart_model_dir)
tokenizer_bart.save_pretrained(bart_model_dir)

print(f"BART model and tokenizer saved to: {bart_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BART model and tokenizer saved to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/bart_base_model1


### Load BART Model (Standalone Session)
Run this cell to load your saved BART model. It includes all necessary imports to start directly from here.

In [27]:
import os
import torch
from transformers import BartTokenizer, BartForConditionalGeneration

# 1. Define the directory path (ensure your Drive is mounted if using Colab)
# Adjust the path below if your save_dir or folder structure differs
save_dir = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/"
bart_model_dir = os.path.join(save_dir, "bart_base_model1")

# 2. Load the objects
print(f"Loading BART model from {bart_model_dir}...")
loaded_tokenizer_bart = BartTokenizer.from_pretrained(bart_model_dir)
loaded_model_bart = BartForConditionalGeneration.from_pretrained(bart_model_dir)

# 3. Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
loaded_model_bart.to(device)

print("BART model and tokenizer loaded successfully.")

Loading BART model from /content/drive/MyDrive/Colab Notebooks/266/266_final_project/bart_base_model1...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

BART model and tokenizer loaded successfully.


In [31]:
# Load decoded predictions and labels from JSON files

preds_input_path = os.path.join(save_dir, "results/bart_preds.json")
labels_input_path = os.path.join(save_dir, "results/bart_labels.json")
decoded_preds_bart, decoded_labels_bart = [],[]
if os.path.exists(preds_input_path) and os.path.exists(labels_input_path):
    with open(preds_input_path, "r", encoding="utf-8") as f:
        loaded_decoded_preds_bart = json.load(f)
    with open(labels_input_path, "r", encoding="utf-8") as f:
        loaded_decoded_labels_bart = json.load(f)
    decoded_preds_bart, decoded_labels_bart  = loaded_decoded_preds_bart, loaded_decoded_labels_bart
    print("BART decoded predictions and labels loaded successfully.")
else:
    print("BART decoded predictions or labels files not found. Please ensure they are saved first.")
    loaded_decoded_preds_bart = []
    loaded_decoded_labels_bart = []

BART decoded predictions and labels loaded successfully.


In [55]:
!pip install -q rouge_score bert_score

In [56]:
import evaluate
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
import nltk

# 1. Ensure METEOR and NLTK dependencies are ready
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)
meteor = evaluate.load('meteor')

from sklearn.model_selection import KFold
import sys
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [57]:
import evaluate
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader

# 1. Load Evaluation Metrics
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')
sacrebleu = evaluate.load('sacrebleu')

# 2. Generation Loop using Loaded Model
def get_model_predictions(model, tokenizer, dataset, batch_size=16):
    model.eval()
    predictions = []
    references = []

    dataloader = DataLoader(dataset, batch_size=batch_size)

    print(f"Generating predictions for {len(dataset)} samples...")
    with torch.no_grad():
        for batch in dataloader:
            inputs = tokenizer(
                batch['encoder_input'],
                padding=True,
                truncation=True,
                max_length=Max_input_length,
                return_tensors="pt"
            ).to(device)

            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=Max_target_length,
                num_beams=4,
                early_stopping=True
            )

            preds = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
            labels = batch['decoder_target']

            predictions.extend(preds)
            references.extend(labels)

    return predictions, references

# Execute generation
decoded_preds_bart, decoded_labels_bart = get_model_predictions(
    loaded_model_bart,
    loaded_tokenizer_bart,
    test_dataset
)

# 3. Compute Metrics
rouge_results_bart = rouge.compute(predictions=decoded_preds_bart, references=decoded_labels_bart)
sacrebleu_results_bart = sacrebleu.compute(predictions=decoded_preds_bart, references=[[r] for r in decoded_labels_bart])

print("\n--- BART Lexical Metrics ---")
for k, v in rouge_results_bart.items():
    print(f"{k}: {v:.4f}")
print(f"sacrebleu: {sacrebleu_results_bart['score']:.4f}")

# 4. Compute Semantic Metrics (BERTScore)
bertscore_results_bart = bertscore.compute(predictions=decoded_preds_bart, references=decoded_labels_bart, lang='en')
print("\n--- BART Semantic Metrics (BERTScore) ---")
print(f"Precision: {np.mean(bertscore_results_bart['precision']):.4f}")
print(f"Recall: {np.mean(bertscore_results_bart['recall']):.4f}")
print(f"F1: {np.mean(bertscore_results_bart['f1']):.4f}")

Generating predictions for 795 samples...

--- BART Lexical Metrics ---
rouge1: 0.2693
rouge2: 0.0898
rougeL: 0.1772
rougeLsum: 0.1771
sacrebleu: 5.4026


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- BART Semantic Metrics (BERTScore) ---
Precision: 0.8752
Recall: 0.8441
F1: 0.8591


### Adding METEOR Evaluation
To compute the METEOR score, we need the `nltk` library and the `meteor` metric from the `evaluate` package.

In [58]:


# 2. Safety check: Regenerate BART predictions if variables were lost during env reset
if 'decoded_preds_bart' not in locals():
    print("Regenerating BART predictions for evaluation...")
    # This assumes trainer_bart and tokenized_test_bart exist from earlier successful cells
    predictions_output_bart = trainer_bart.predict(tokenized_test_bart)

    label_ids = predictions_output_bart.label_ids
    label_ids = np.where(label_ids != -100, label_ids, tokenizer_bart.pad_token_id)

    preds = predictions_output_bart.predictions
    preds = np.where(preds != -100, preds, tokenizer_bart.pad_token_id)

    decoded_preds_bart = tokenizer_bart.batch_decode(preds, skip_special_tokens=True)
    decoded_labels_bart = tokenizer_bart.batch_decode(label_ids, skip_special_tokens=True)

# 3. Compute METEOR
meteor_results = meteor.compute(predictions=decoded_preds_bart, references=decoded_labels_bart)
print(f"\n--- Final BART Evaluation ---")
print(f"METEOR Score: {meteor_results['meteor']:.4f}")


--- Final BART Evaluation ---
METEOR Score: 0.1726


### Post-hoc K-Fold Cross Validation on Test Set
This section splits the existing test set into K folds, evaluates each fold independently, and reports the mean and standard deviation of the metrics.

In [59]:
from sklearn.model_selection import KFold
import numpy as np
import sys
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments

def get_eval_trainer(model, tokenizer):
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    return Seq2SeqTrainer(
        model=model,
        args=Seq2SeqTrainingArguments(
            output_dir="./temp_eval",
            predict_with_generate=True,
            fp16=torch.cuda.is_available(),
            per_device_eval_batch_size=8,
            report_to="none"
        ),
        data_collator=data_collator,
        processing_class=tokenizer
    )

def run_kfold_evaluation(dataset, model, tokenizer, k=5):
    eval_trainer = get_eval_trainer(model, tokenizer)
    kf = KFold(n_splits=k, shuffle=True, random_state=initseed)
    all_results = []
    indices = np.arange(len(dataset))

    print(f"Starting optimized {k}-fold cross-validation with SacreBLEU...\n")

    for fold, (_, fold_indices) in enumerate(kf.split(indices)):
        fold_dataset = dataset.select(fold_indices)
        print(f"Fold {fold+1}/{k} processing:")

        # Use Trainer's predict
        output = eval_trainer.predict(fold_dataset)

        # Same-line progress feedback
        sys.stdout.write(f"\rFold {fold+1} Status: [DONE] 100% complete")
        sys.stdout.flush()
        print("\nComputing metrics...")

        label_ids = np.where(output.label_ids != -100, output.label_ids, tokenizer.pad_token_id)
        preds = np.where(output.predictions != -100, output.predictions, tokenizer.pad_token_id)

        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        r = rouge.compute(predictions=decoded_preds, references=decoded_labels)
        sb = sacrebleu.compute(predictions=decoded_preds, references=[[r] for r in decoded_labels])
        bs = bertscore.compute(predictions=decoded_preds, references=decoded_labels, lang='en')
        m = meteor.compute(predictions=decoded_preds, references=decoded_labels)

        fold_metrics = {
            'rougeL': r['rougeL'],
            'sacrebleu': sb['score'],
            'bert_f1': np.mean(bs['f1']),
            'meteor': m['meteor']
        }

        all_results.append(fold_metrics)
        print(f"Fold {fold+1} Results -> SacreBLEU: {fold_metrics['sacrebleu']:.4f} | METEOR: {fold_metrics['meteor']:.4f}\n")

    metrics_summary = {}
    for key in all_results[0].keys():
        values = [res[key] for res in all_results]
        metrics_summary[key] = {'mean': np.mean(values), 'std': np.std(values)}

    return metrics_summary

# Run the optimized evaluation
bart_kfold_summary = run_kfold_evaluation(tokenized_test_bart, loaded_model_bart, loaded_tokenizer_bart, k=5)

Starting optimized 5-fold cross-validation with SacreBLEU...

Fold 1/5 processing:


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1638: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Fold 1 Status: [DONE] 100% complete
Computing metrics...
Fold 1 Results -> SacreBLEU: 0.0182 | METEOR: 0.0556

Fold 2/5 processing:


Fold 2 Status: [DONE] 100% complete
Computing metrics...
Fold 2 Results -> SacreBLEU: 0.0142 | METEOR: 0.0593

Fold 3/5 processing:


Fold 3 Status: [DONE] 100% complete
Computing metrics...
Fold 3 Results -> SacreBLEU: 0.0126 | METEOR: 0.0558

Fold 4/5 processing:


Fold 4 Status: [DONE] 100% complete
Computing metrics...
Fold 4 Results -> SacreBLEU: 0.0138 | METEOR: 0.0571

Fold 5/5 processing:


Fold 5 Status: [DONE] 100% complete
Computing metrics...
Fold 5 Results -> SacreBLEU: 0.0183 | METEOR: 0.0551



In [60]:
print("\n--- BART K-Fold Cross-Validation Summary ---")
for metric, stats in bart_kfold_summary.items():
    name = "SACREBLEU" if metric == "sacrebleu" else metric.upper()
    print(f"{name}: {stats['mean']:.5f} (+/- {stats['std']:.5f})")


--- BART K-Fold Cross-Validation Summary ---
ROUGEL: 0.09865 (+/- 0.00258)
SACREBLEU: 0.01539 (+/- 0.00237)
BERT_F1: 0.84309 (+/- 0.00142)
METEOR: 0.05658 (+/- 0.00153)


### STS Evaluation with DistilGPT2
This section uses a decoder-only model (DistilGPT2) to compute Semantic Textual Similarity (STS) between the generated reports and the reference reports.

In [61]:
from transformers import AutoModel, AutoTokenizer
from torch.nn.functional import cosine_similarity

# 1. Load DistilGPT2 for STS
sts_model_name = "distilgpt2"
sts_tokenizer = AutoTokenizer.from_pretrained(sts_model_name)
sts_model = AutoModel.from_pretrained(sts_model_name).to(device)
sts_tokenizer.pad_token = sts_tokenizer.eos_token

def get_embeddings(text_list, model, tokenizer):
    model.eval()
    inputs = tokenizer(text_list, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        # Use mean pooling of hidden states as sentence representation
        embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

def compute_sts_score(predictions, references, batch_size=16):
    scores = []
    print(f"Computing STS scores for {len(predictions)} samples...")

    for i in range(0, len(predictions), batch_size):
        batch_preds = predictions[i:i+batch_size]
        batch_refs = references[i:i+batch_size]

        emb_preds = get_embeddings(batch_preds, sts_model, sts_tokenizer)
        emb_refs = get_embeddings(batch_refs, sts_model, sts_tokenizer)

        # Compute cosine similarity
        sim = cosine_similarity(emb_preds, emb_refs)
        scores.extend(sim.cpu().tolist())

    return np.mean(scores)

# 2. Run STS Evaluation
bart_sts_score = compute_sts_score(decoded_preds_bart, decoded_labels_bart)
print(f"\nBART DistilGPT2 STS Score: {bart_sts_score:.4f}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Computing STS scores for 795 samples...

BART DistilGPT2 STS Score: 0.9911


In [34]:
import json
import os

# Define the output path for the BART baseline predictions
bart_baseline_output_path = os.path.join(save_dir,
                          "results/baseline/test_baseline_bart.json")

# Ensure the directory exists
os.makedirs(os.path.dirname(bart_baseline_output_path), exist_ok=True)

# Prepare the data in the required format
# Each record should have 'ground_truth_report', 'baseline_report', 'market', and 'date'
# Assuming decoded_labels_bart and decoded_preds_bart are aligned with test_df

bart_baseline_records = []
for idx, (ref, pred) in enumerate(zip(decoded_labels_bart, decoded_preds_bart)):
    original_record = test_df.iloc[idx] # Get the original row from test_df
    bart_baseline_records.append({
        "ground_truth_report": ref,
        "baseline_report": pred,
        "market": original_record["market"],
        "date": str(original_record["date"])      # Convert Timestamp to string
    })

# Save the data to a JSON file
with open(bart_baseline_output_path, "w", encoding="utf-8") as f:
    json.dump(bart_baseline_records, f, indent=4)

print(f"BART baseline predictions saved to: {bart_baseline_output_path}")

BART baseline predictions saved to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/results/baseline/test_baseline_bart.json


In [63]:
# Save decoded predictions and labels to JSON files
preds_output_path = os.path.join(save_dir, "results/bart_preds.json")
labels_output_path = os.path.join(save_dir, "results/bart_labels.json")

os.makedirs(os.path.dirname(preds_output_path), exist_ok=True)

with open(preds_output_path, "w", encoding="utf-8") as f:
    json.dump(decoded_preds_bart, f, indent=4)

with open(labels_output_path, "w", encoding="utf-8") as f:
    json.dump(decoded_labels_bart, f, indent=4)

print(f"BART decoded predictions saved to: {preds_output_path}")
print(f"BART decoded labels saved to: {labels_output_path}")

BART decoded predictions saved to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/results/bart_preds.json
BART decoded labels saved to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/results/bart_labels.json
